# UMUD — Train Fasc U-Net from Prep Dataset

**GPU notebook** — mounts a **prep notebook** dataset (`dataset_sources`). No competition TIFF scan or inline align.

BirdCLEF pattern: `multilabel-234-v2` mounts `species-v2-*` datasets and uses `get_image_files`.

> Edit *Configuration*, then re-run from there downward.


## Configuration


In [ ]:
# --- Parameters you can change ---
RANDOM_SEED = 42
TRAIN_RUN = 1  # 1 = mount timing-50, 2 = mount timing-200

VALID_PCT = 0.20
BATCH_SIZE = 8
EPOCHS = 1
ARCH = "resnet34"
IMG_SIZE = 256  # must match prep dataset

TRAIN_PROFILES = {
    1: {
        "dataset_slug": "ucheozoemena/umud-aligned-fasc-timing-50",
        "mount_name": "umud-aligned-fasc-timing-50",
        "label": "T1 fasc 50×1ep",
    },
    2: {
        "dataset_slug": "ucheozoemena/umud-aligned-fasc-timing-200",
        "mount_name": "umud-aligned-fasc-timing-200",
        "label": "T2 fasc 200×1ep",
    },
}

profile = TRAIN_PROFILES[TRAIN_RUN]
DATASET_SLUG = profile["dataset_slug"]
MOUNT_NAME = profile["mount_name"]
print(f"TRAIN_RUN={TRAIN_RUN} | {profile['label']} | dataset={DATASET_SLUG}")


In [ ]:
from __future__ import annotations

import random
import time
from pathlib import Path

import pandas as pd
from fastai.vision.all import *
from fastai.vision.core import PILImage, PILMask
from fastai.vision.data import ImageBlock, MaskBlock, TransformBlock
from fastai.vision.augment import Resize
from fastai.layers import CrossEntropyLossFlat

DATASET_ROOT = Path(f"/kaggle/input/datasets/{DATASET_SLUG}")
IMG_DIR = DATASET_ROOT / "images"
MSK_DIR = DATASET_ROOT / "masks"
WORKING = Path("/kaggle/working")

print(f"Dataset root exists: {DATASET_ROOT.exists()}")
for p in [IMG_DIR, MSK_DIR]:
    print(f"  {p.name}: exists={p.exists()}")


In [ ]:
def open_image_pil(fn):
    return PILImage.create(fn)

def open_mask_pil(fn):
    return PILMask.create(fn, mode="L")

def make_dls(fnames, valid_pct=0.20, bs=8, seed=42):
    block = DataBlock(
        blocks=(TransformBlock(type_tfms=open_image_pil), TransformBlock(type_tfms=open_mask_pil)),
        get_items=lambda _: fnames,
        get_x=lambda f: IMG_DIR / f.name,
        get_y=lambda f: MSK_DIR / f.name,
        splitter=RandomSplitter(valid_pct=valid_pct, seed=seed),
        item_tfms=Resize(IMG_SIZE),
        batch_tfms=[*aug_transforms(size=IMG_SIZE, min_scale=0.75), Normalize.from_stats(*imagenet_stats)],
    )
    return block.dataloaders(fnames, bs=bs, num_workers=2)

img_fnames = get_image_files(IMG_DIR)
msk_lookup = {p.name for p in get_image_files(MSK_DIR)}
fnames = [f for f in img_fnames if f.name in msk_lookup]
print(f"Pairs: {len(fnames)}")
assert len(fnames) > 0, "No image/mask pairs in mounted dataset"


In [ ]:
t0 = time.perf_counter()
dls = make_dls(fnames, valid_pct=VALID_PCT, bs=BATCH_SIZE, seed=RANDOM_SEED)
dls.show_batch(max_n=4)


In [ ]:
learn = unet_learner(
    dls,
    ARCH,
    metrics=[Dice()],
    loss_func=CrossEntropyLossFlat(axis=1),
)
learn.fine_tune(EPOCHS)
t1 = time.perf_counter()
train_sec = t1 - t0
print(f"Train wall-clock: {train_sec:.1f}s ({train_sec / max(1, len(fnames)) / max(1, EPOCHS):.2f}s/pair/epoch)")

learn.export(WORKING / "fasc_baseline.pkl")

timing = pd.DataFrame(
    [
        {
            "train_run": TRAIN_RUN,
            "n_pairs": len(fnames),
            "epochs": EPOCHS,
            "img_size": IMG_SIZE,
            "total_sec": round(train_sec, 1),
            "sec_per_pair_epoch": round(train_sec / max(1, len(fnames) * EPOCHS), 3),
            "dataset": DATASET_SLUG,
        }
    ]
)
timing.to_csv(WORKING / "timing_report.csv", index=False)
display(timing)
